In [1]:
import logging
from pathlib import Path

import pandas as pd
import numpy as np

In [2]:
logging.basicConfig(
    level=logging.INFO,
    filename='students.log',
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

DATA_DIR = Path('.')
MESSY_FILE = DATA_DIR / 'messy_students.csv'
CLEAN_FILE = DATA_DIR / 'clean_students.csv'

def load_csv_file(file_path):
    try:
        if not file_path.exists():
            raise FileNotFoundError(f'{file_path} does not exist')
        logger.info('Loading data from %s', file_path)
        return pd.read_csv(file_path)
    except Exception as exc:
        logger.exception('Failed to load CSV file')
        raise

def save_csv_file(dataframe, file_path):
    try:
        dataframe.to_csv(file_path, index=False)
        logger.info('Saved data to %s', file_path)
    except Exception:
        logger.exception('Failed to save CSV file')
        raise

In [3]:
# Creating messy Nepali-style student dataset manually

data = {
    "name": [" Aarav ", "sita", "BIKASH", "priya ", "Kiran", None,
             " anjali", "ROSHAN", "Mina ", "suman", "Aarav",
             "  bibek ", "NISHA", "gopal", "Rekha ", "sunita",
             "KABITA", "ram ", "Hari", "  sita "],

    "score": ["88", " 76 ", None, "-12", "91", "67",
              "45", "101", "59", "0", "88",
              " 72", "-5", "83", "95 ", None,
              "64", "39", "110", "76"]
}

# Create DataFrame
df_messy = pd.DataFrame(data)

# Add duplicate rows deliberately
df_messy = pd.concat([df_messy, df_messy.iloc[[1, 4, 10]]], ignore_index=True)

# Save to CSV
try:
    save_csv_file(df_messy, MESSY_FILE)
    logger.info('Messy dataset created successfully')
except Exception:
    print('Could not create messy_students.csv.')

print("messy_students.csv created successfully!")
print(df_messy)

messy_students.csv created successfully!
        name score
0     Aarav     88
1       sita   76 
2     BIKASH   NaN
3     priya    -12
4      Kiran    91
5        NaN    67
6     anjali    45
7     ROSHAN   101
8      Mina     59
9      suman     0
10     Aarav    88
11    bibek     72
12     NISHA    -5
13     gopal    83
14    Rekha    95 
15    sunita   NaN
16    KABITA    64
17      ram     39
18      Hari   110
19     sita     76
20      sita   76 
21     Kiran    91
22     Aarav    88


In [4]:
# Load messy dataset

try:
    df = load_csv_file(MESSY_FILE)
    logger.info('Dataset loaded successfully')
    display(df.head(10))
except Exception:
    df = pd.DataFrame()
    print('Dataset could not be loaded. Check the logs for details.')

,name,score
0,Aarav,88.0
1,sita,76.0
2,BIKASH,NaN
3,priya,-12.0
4,Kiran,91.0
5,NaN,67.0
6,anjali,45.0
7,ROSHAN,101.0
8,Mina,59.0
9,suman,0.0


In [5]:
# Basic information about dataset

print("====DATAFRAME INFO====" )
print(df.info())

print("\n===== NULL VALUES =====")
print(df.isnull().sum())


====DATAFRAME INFO====
<class 'pandas.DataFrame'>
RangeIndex: 23 entries, 0 to 22
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   name    22 non-null     str    
 1   score   21 non-null     float64
dtypes: float64(1), str(1)
memory usage: 500.0 bytes
None

===== NULL VALUES =====
name     1
score    2
dtype: int64


In [6]:
# Find duplicate rows

duplicates = df[df.duplicated()]

print("===== DUPLICATE ROWS =====")
print(duplicates)

print(f"\nTotal Duplicate Rows: {df.duplicated().sum()}")

===== DUPLICATE ROWS =====
     name  score
20   sita   76.0
21  Kiran   91.0
22  Aarav   88.0

Total Duplicate Rows: 3


In [7]:
# Store original row count before cleaning

before_rows = df.shape[0]

print(f"Rows Before Cleaning: {before_rows}")

Rows Before Cleaning: 23


In [8]:
try:
    # Validate column existence first
    if 'name' not in df.columns:
        raise KeyError("Expected a 'name' column in the dataset")

    # Clean whitespace safely
    df["name"] = df["name"].astype("string").str.strip()

    logger.info("Whitespace cleaned successfully")
    
    df["name"] = df["name"].str.title()

    logging.info("Name casing standardized!")

except Exception as e:
    logger.exception(f"Unexpected error while cleaning names: {e}")
    raise

In [10]:
try:
    # Validate column existence
    if 'score' not in df.columns:
        raise KeyError("Expected a 'score' column in the dataset")

    # Convert to numeric safely
    df["score"] = pd.to_numeric(df["score"], errors="coerce")

    # Optional: detect how much data got corrupted during conversion
    null_count = df["score"].isna().sum()
    if null_count > 0:
        logger.warning(f"{null_count} values could not be converted and became NaN")

    logger.info("Score column converted to numeric successfully")
    print(df.dtypes)


except Exception as e:
    logger.exception(f"Unexpected error while converting score column: {e}")
    raise

name      string
score    float64
dtype: object


In [11]:
# Fill missing names with 'Unknown'
df["name"] = df["name"].fillna("Unknown")

# Fill missing scores with median score
median_score = df["score"].median()

df["score"] = df["score"].fillna(median_score)

print("Missing values handled successfully!")

Missing values handled successfully!


In [12]:
# Remove rows where score is below 0 or above 100

df = df[(df["score"] >= 0) & (df["score"] <= 100)]

print("Invalid scores removed!")

Invalid scores removed!


In [13]:
# Remove duplicate rows

df = df.drop_duplicates()

print("Duplicate rows removed!")

Duplicate rows removed!


In [14]:
# Function to assign grades

def assign_grade(score):
    if pd.isna(score):
        return 'F'
    if score >= 90:
        return "A"
    elif score >= 75:
        return "B"
    elif score >= 50:
        return "C"
    else:
        return "F"


# Create grade column
df["grade"] = df["score"].apply(assign_grade)

print("Grade column added successfully!")

Grade column added successfully!


In [15]:
print("===== CLEANED DATASET =====")

df

===== CLEANED DATASET =====


,name,score,grade
0,Aarav,88.0,B
1,Sita,76.0,B
2,Bikash,76.0,B
4,Kiran,91.0,A
5,Unknown,67.0,C
6,Anjali,45.0,F
8,Mina,59.0,C
9,Suman,0.0,F
11,Bibek,72.0,C
13,Gopal,83.0,B


In [16]:
# Save cleaned data to CSV

save_csv_file(df, CLEAN_FILE)
print("clean_students.csv saved successfully!")

clean_students.csv saved successfully!


In [17]:
after_rows = df.shape[0]

print("===== ROW COUNT COMPARISON =====")

print(f"Rows Before Cleaning : {before_rows}")
print(f"Rows After Cleaning  : {after_rows}")
print(f"Rows Removed         : {before_rows - after_rows}")

===== ROW COUNT COMPARISON =====
Rows Before Cleaning : 23
Rows After Cleaning  : 14
Rows Removed         : 9
